In [1]:
import os
import json
import urllib.request

REPO_URL = "https://github.com/glamouroussage/Rayna-Assignment"
print("Notebook is ready!")
print("Repository:", REPO_URL)

Notebook is ready!
Repository: https://github.com/glamouroussage/Rayna-Assignment


In [2]:

import json
import urllib.request

BASE_URL = (
    "https://raw.githubusercontent.com/"
    "glamouroussage/Rayna-Assignment/main/"
)

for filename in ["comments.json", "comments-v2.json"]:
    url = BASE_URL + filename

    with urllib.request.urlopen(url) as response:
        data = json.load(response)

    print(f"\nFile: {filename}")
    print("Top-level keys:", list(data.keys()))
    print("Number of comments:", len(data["comments"]))
    print("First comment:", data["comments"][0])



File: comments.json
Top-level keys: ['course', 'forum', 'note', 'comments']
Number of comments: 100
First comment: {'id': 1, 'author': 'Aditya Kulkarni', 'thread': 'Doubts', 'posted_at': '2026-10-12 09:14', 'comment': "Ma'am, is it okay if we use Claude for the frontend and Codex for the backend logic, or should the whole artifact be built with one tool?"}

File: comments-v2.json
Top-level keys: ['course', 'forum', 'note', 'comments']
Number of comments: 100
First comment: {'id': 1, 'author': 'Arnav Deshmukh', 'thread': 'Doubts', 'posted_at': '2026-11-02 09:05', 'comment': "Sir, for assignment 4 can we evaluate the free Gemini or does it have to be ChatGPT? I don't have a paid plan for either."}


In [ ]:
acc id- 7dac36566e6605a80b2134b40a134de2
api token- 

In [3]:

import getpass
import json
import urllib.request
import urllib.error

ACCOUNT_ID = getpass.getpass("Enter your Cloudflare Account ID: ")
API_TOKEN = getpass.getpass("Enter your Cloudflare API token: ")

print("Credentials entered. Values will not be printed.")


Enter your Cloudflare Account ID: ··········
Enter your Cloudflare API token: ··········
Credentials entered. Values will not be printed.


In [4]:

url = (
    f"https://api.cloudflare.com/client/v4/accounts/"
    f"{ACCOUNT_ID}/ai/run/@cf/cloudflare/clef-flash"
)

payload = {
    "state": {
        "comment": "Thank you, this lesson was really helpful!"
    },
    "questions": {
        "moderation": {
            "type": "choice",
            "instructions": (
                "Classify the student forum comment as "
                "acceptable, toxic, or spam."
            ),
            "criteria": {
                "acceptable": (
                    "A legitimate, relevant, non-abusive comment."
                ),
                "toxic": (
                    "Abusive, insulting, threatening, or "
                    "harassing content."
                ),
                "spam": (
                    "Unwanted promotional, repetitive, or "
                    "irrelevant content."
                )
            }
        }
    }
}

request = urllib.request.Request(
    url,
    data=json.dumps(payload).encode("utf-8"),
    method="POST",
    headers={
        "Authorization": f"Bearer {API_TOKEN}",
        "Content-Type": "application/json"
    }
)

try:
    with urllib.request.urlopen(request, timeout=60) as response:
        result = json.load(response)

    if result.get("success"):
        answer = result["result"]["answers"]["moderation"]
        print("Model:", result["result"]["model"])
        print("Classification:", answer["choice"])
        print("Probabilities:", answer["probabilities"])
        print("Confidence:", answer["confidence"])
    else:
        print("API returned an error:", result.get("errors"))

except urllib.error.HTTPError as e:
    print("HTTP error:", e.code)
    print(e.read().decode("utf-8")[:1500])
except Exception as e:
    print("Error:", str(e))


Model: clef-flash
Classification: acceptable
Probabilities: {'acceptable': 0.9406, 'toxic': 0.0295, 'spam': 0.0299}
Confidence: 0.8298


In [5]:

import json
import time
import html
import urllib.request
import urllib.error
from pathlib import Path
from IPython.display import display, HTML

# Load the original dataset
with urllib.request.urlopen(
    BASE_URL + "comments.json"
) as response:
    dataset = json.load(response)

comments = dataset["comments"]
results = []

# Define the classification criteria
criteria = {
    "acceptable": (
        "A legitimate course-related question, answer, discussion, "
        "feedback, or ordinary opinion without abusive targeting."
    ),
    "toxic": (
        "Insults, harassment, threats, discriminatory attacks, "
        "or abusive personal targeting. Criticism alone is not toxic."
    ),
    "spam": (
        "Unsolicited advertising, scams, repetitive promotion, "
        "or irrelevant commercial solicitation."
    )
}

url = (
    f"https://api.cloudflare.com/client/v4/accounts/"
    f"{ACCOUNT_ID}/ai/run/@cf/cloudflare/clef-flash"
)

for i, comment in enumerate(comments, start=1):
    payload = {
        "state": {"comment": comment["comment"]},
        "questions": {
            "moderation": {
                "type": "choice",
                "instructions": (
                    "Classify this student forum comment as exactly "
                    "one of acceptable, toxic, or spam. "
                    "Judge the comment itself in context. "
                    "Quoted abusive language used to report misconduct "
                    "is not necessarily toxic. Promotional scams are spam."
                ),
                "criteria": criteria
            }
        }
    }

    request = urllib.request.Request(
        url,
        data=json.dumps(payload).encode("utf-8"),
        method="POST",
        headers={
            "Authorization": f"Bearer {API_TOKEN}",
            "Content-Type": "application/json"
        }
    )

    # Retry temporary rate-limit/server errors
    for attempt in range(4):
        try:
            with urllib.request.urlopen(
                request, timeout=60
            ) as response:
                data = json.load(response)

            if not data.get("success"):
                raise RuntimeError(str(data.get("errors")))

            answer = data["result"]["answers"]["moderation"]

            results.append({
                **comment,
                "classification": answer["choice"],
                "probabilities": answer["probabilities"],
                "confidence": answer["confidence"],
                "model": data["result"]["model"],
                "error": None
            })
            break

        except urllib.error.HTTPError as e:
            if e.code in (429, 500, 502, 503, 504) and attempt < 3:
                time.sleep(2 ** (attempt + 1))
                continue

            results.append({
                **comment,
                "classification": "ERROR",
                "probabilities": {},
                "confidence": None,
                "model": "clef-flash",
                "error": f"HTTP {e.code}: {e.read().decode()[:300]}"
            })
            break

        except Exception as e:
            if attempt < 3:
                time.sleep(2 ** (attempt + 1))
                continue

            results.append({
                **comment,
                "classification": "ERROR",
                "probabilities": {},
                "confidence": None,
                "model": "clef-flash",
                "error": str(e)[:300]
            })

    print(f"Processed {i}/{len(comments)}")
    time.sleep(0.3)

# Save raw model outputs for reproducibility
Path("results.json").write_text(
    json.dumps(results, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

# Create a scan-friendly HTML report
rows = []
for r in results:
    probs = " | ".join(
        f"{k}: {v:.1%}" for k, v in r["probabilities"].items()
    )
    confidence = (
        f"{r['confidence']:.1%}"
        if r["confidence"] is not None else "N/A"
    )

    rows.append(f"""
    <tr>
      <td>{r['id']}</td>
      <td>{html.escape(r['author'])}</td>
      <td>{html.escape(r['thread'])}</td>
      <td>{html.escape(r['comment'])}</td>
      <td><strong>{html.escape(r['classification'])}</strong></td>
      <td>{html.escape(probs)}</td>
      <td>{confidence}</td>
    </tr>
    """)

report = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Clef Flash — Comment Moderation Results</title>
<style>
body {{ font: 15px Arial, sans-serif; margin: 24px; color: #222; }}
h1 {{ margin-bottom: 8px; }}
.note {{ color: #555; }}
input {{ padding: 10px; width: min(420px, 90%); margin: 12px 0; }}
table {{ border-collapse: collapse; width: 100%; }}
th, td {{ border: 1px solid #ddd; padding: 10px; text-align: left; }}
th {{ background: #f1f3f5; position: sticky; top: 0; }}
tr:nth-child(even) {{ background: #fafafa; }}
td:nth-child(4) {{ min-width: 320px; white-space: pre-wrap; }}
</style>
</head>
<body>
<h1>Clef Flash — Comment Moderation</h1>
<p class="note">Dataset: comments.json | Model: clef-flash |
Comments: {len(results)} | Errors: {sum(r['classification'] == 'ERROR' for r in results)}</p>
<p class="note">Review every classification manually. Model confidence is
not a guarantee of correctness.</p>
<input id="filter" placeholder="Filter by author, comment, or label..."
oninput="filterRows()">
<table id="results">
<thead><tr>
<th>ID</th><th>Author</th><th>Thread</th><th>Comment</th>
<th>Classification</th><th>Probabilities</th><th>Confidence</th>
</tr></thead>
<tbody>{''.join(rows)}</tbody>
</table>
<script>
function filterRows() {{
  const q = document.getElementById('filter').value.toLowerCase();
  document.querySelectorAll('#results tbody tr').forEach(row => {{
    row.style.display = row.innerText.toLowerCase().includes(q) ? '' : 'none';
  }});
}}
</script>
</body>
</html>"""

Path("results.html").write_text(report, encoding="utf-8")
display(HTML(
    f"<h3>Finished: {len(results)} comments processed</h3>"
    f"<p>Errors: {sum(r['classification'] == 'ERROR' for r in results)}</p>"
    "<p>Files created: results.html and results.json</p>"
))


Processed 1/100
Processed 2/100
Processed 3/100
Processed 4/100
Processed 5/100
Processed 6/100
Processed 7/100
Processed 8/100
Processed 9/100
Processed 10/100
Processed 11/100
Processed 12/100
Processed 13/100
Processed 14/100
Processed 15/100
Processed 16/100
Processed 17/100
Processed 18/100
Processed 19/100
Processed 20/100
Processed 21/100
Processed 22/100
Processed 23/100
Processed 24/100
Processed 25/100
Processed 26/100
Processed 27/100
Processed 28/100
Processed 29/100
Processed 30/100
Processed 31/100
Processed 32/100
Processed 33/100
Processed 34/100
Processed 35/100
Processed 36/100
Processed 37/100
Processed 38/100
Processed 39/100
Processed 40/100
Processed 41/100
Processed 42/100
Processed 43/100
Processed 44/100
Processed 45/100
Processed 46/100
Processed 47/100
Processed 48/100
Processed 49/100
Processed 50/100
Processed 51/100
Processed 52/100
Processed 53/100
Processed 54/100
Processed 55/100
Processed 56/100
Processed 57/100
Processed 58/100
Processed 59/100
Proces

In [6]:

import json
import time
import html
import urllib.request
import urllib.error
from pathlib import Path
from IPython.display import display, HTML

# Load the harder dataset
with urllib.request.urlopen(BASE_URL + "comments-v2.json") as response:
    dataset_v2 = json.load(response)

comments_v2 = dataset_v2["comments"]
results_v2 = []

# More explicit criteria for difficult and borderline cases
criteria_v2 = {
    "acceptable": (
        "Legitimate course-related discussion, questions, answers, "
        "criticism, disagreement, humour, or feedback. A comment is "
        "not toxic merely because it is negative or disagrees with someone."
    ),
    "toxic": (
        "Personal abuse, targeted insults, harassment, threats, "
        "or discriminatory attacks. Criticism of an idea or course "
        "is not toxic unless it becomes abusive."
    ),
    "spam": (
        "Unsolicited advertising, scams, irrelevant promotions, "
        "repeated commercial solicitation, or messages intended "
        "primarily to promote unrelated products or services."
    )
}

url_v2 = (
    f"https://api.cloudflare.com/client/v4/accounts/"
    f"{ACCOUNT_ID}/ai/run/@cf/cloudflare/clef"
)

for i, comment in enumerate(comments_v2, start=1):
    payload = {
        "state": {"comment": comment["comment"]},
        "questions": {
            "moderation": {
                "type": "choice",
                "instructions": (
                    "Classify the student forum comment as exactly one "
                    "of acceptable, toxic, or spam. Use the comment's "
                    "meaning and context. Do not confuse strong criticism "
                    "with abuse, or quoted abusive language with an attack "
                    "by the author. Choose spam for irrelevant promotions."
                ),
                "criteria": criteria_v2
            }
        }
    }

    request = urllib.request.Request(
        url_v2,
        data=json.dumps(payload).encode("utf-8"),
        method="POST",
        headers={
            "Authorization": f"Bearer {API_TOKEN}",
            "Content-Type": "application/json"
        }
    )

    for attempt in range(4):
        try:
            with urllib.request.urlopen(request, timeout=60) as response:
                data = json.load(response)

            if not data.get("success"):
                raise RuntimeError(str(data.get("errors")))

            answer = data["result"]["answers"]["moderation"]

            results_v2.append({
                **comment,
                "classification": answer["choice"],
                "probabilities": answer["probabilities"],
                "confidence": answer["confidence"],
                "model": data["result"]["model"],
                "error": None
            })
            break

        except urllib.error.HTTPError as e:
            message = e.read().decode("utf-8")[:300]

            if e.code in (429, 500, 502, 503, 504) and attempt < 3:
                time.sleep(2 ** (attempt + 1))
                continue

            results_v2.append({
                **comment,
                "classification": "ERROR",
                "probabilities": {},
                "confidence": None,
                "model": "clef",
                "error": f"HTTP {e.code}: {message}"
            })
            break

        except Exception as e:
            if attempt < 3:
                time.sleep(2 ** (attempt + 1))
                continue

            results_v2.append({
                **comment,
                "classification": "ERROR",
                "probabilities": {},
                "confidence": None,
                "model": "clef",
                "error": str(e)[:300]
            })

    print(f"Processed {i}/{len(comments_v2)}")
    time.sleep(0.3)

# Save the model outputs
Path("results-v2.json").write_text(
    json.dumps(results_v2, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

# Build the HTML report
rows = []

for r in results_v2:
    probs = " | ".join(
        f"{k}: {v:.1%}"
        for k, v in r["probabilities"].items()
    )

    confidence = (
        f"{r['confidence']:.1%}"
        if r["confidence"] is not None else "N/A"
    )

    rows.append(f"""
    <tr>
      <td>{html.escape(str(r.get('id', '')))}</td>
      <td>{html.escape(str(r.get('author', '')))}</td>
      <td>{html.escape(str(r.get('thread', '')))}</td>
      <td>{html.escape(str(r.get('comment', '')))}</td>
      <td><strong>{html.escape(r['classification'])}</strong></td>
      <td>{html.escape(probs)}</td>
      <td>{confidence}</td>
    </tr>
    """)

errors_v2 = sum(
    r["classification"] == "ERROR" for r in results_v2
)

report_v2 = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Clef — Harder Comment Moderation Results</title>
<style>
body {{ font: 15px Arial, sans-serif; margin: 24px; color: #222; }}
h1 {{ margin-bottom: 8px; }}
.note {{ color: #555; }}
input {{ padding: 10px; width: min(420px, 90%); margin: 12px 0; }}
table {{ border-collapse: collapse; width: 100%; }}
th, td {{ border: 1px solid #ddd; padding: 10px; text-align: left; }}
th {{ background: #f1f3f5; position: sticky; top: 0; }}
tr:nth-child(even) {{ background: #fafafa; }}
td:nth-child(4) {{ min-width: 320px; white-space: pre-wrap; }}
</style>
</head>
<body>
<h1>Clef — Harder Comment Moderation</h1>
<p class="note">Dataset: comments-v2.json | Model: clef |
Comments: {len(results_v2)} | Errors: {errors_v2}</p>
<p class="note">Review every classification manually. Confidence is not
a guarantee of correctness.</p>
<input id="filter" placeholder="Filter by author, comment, or label..."
oninput="filterRows()">
<table id="results">
<thead><tr>
<th>ID</th><th>Author</th><th>Thread</th><th>Comment</th>
<th>Classification</th><th>Probabilities</th><th>Confidence</th>
</tr></thead>
<tbody>{''.join(rows)}</tbody>
</table>
<script>
function filterRows() {{
  const q = document.getElementById('filter').value.toLowerCase();
  document.querySelectorAll('#results tbody tr').forEach(row => {{
    row.style.display = row.innerText.toLowerCase().includes(q) ? '' : 'none';
  }});
}}
</script>
</body>
</html>"""

Path("results-v2.html").write_text(
    report_v2, encoding="utf-8"
)

display(HTML(
    f"<h3>Finished: {len(results_v2)} comments processed</h3>"
    f"<p>Model: clef | Errors: {errors_v2}</p>"
    "<p>Created results-v2.html and results-v2.json</p>"
))


Processed 1/100
Processed 2/100
Processed 3/100
Processed 4/100
Processed 5/100
Processed 6/100
Processed 7/100
Processed 8/100
Processed 9/100
Processed 10/100
Processed 11/100
Processed 12/100
Processed 13/100
Processed 14/100
Processed 15/100
Processed 16/100
Processed 17/100
Processed 18/100
Processed 19/100
Processed 20/100
Processed 21/100
Processed 22/100
Processed 23/100
Processed 24/100
Processed 25/100
Processed 26/100
Processed 27/100
Processed 28/100
Processed 29/100
Processed 30/100
Processed 31/100
Processed 32/100
Processed 33/100
Processed 34/100
Processed 35/100
Processed 36/100
Processed 37/100
Processed 38/100
Processed 39/100
Processed 40/100
Processed 41/100
Processed 42/100
Processed 43/100
Processed 44/100
Processed 45/100
Processed 46/100
Processed 47/100
Processed 48/100
Processed 49/100
Processed 50/100
Processed 51/100
Processed 52/100
Processed 53/100
Processed 54/100
Processed 55/100
Processed 56/100
Processed 57/100
Processed 58/100
Processed 59/100
Proces

In [7]:
from IPython.display import display, HTML
from pathlib import Path

display(HTML(Path("results.html").read_text(encoding="utf-8")))

ID,Author,Thread,Comment,Classification,Probabilities,Confidence
1,Aditya Kulkarni,Doubts,"Ma'am, is it okay if we use Claude for the frontend and Codex for the backend logic, or should the whole artifact be built with one tool?",acceptable,acceptable: 95.4% | toxic: 2.4% | spam: 2.2%,86.7%
2,Sneha Iyer,Share your artifacts,Submitted my artifact: a timetable clash checker for our dept. Claude made the UI in like 10 mins but I spent 2 hours fixing the CSV parsing. Link is in my submission.,acceptable,acceptable: 91.5% | toxic: 3.5% | spam: 5.0%,76.1%
3,AssignmentGuru_99,General,"🔥🔥 GET YOUR ASSIGNMENTS DONE BY EXPERTS 🔥🔥 Claude/Codex/ML projects, plagiarism free, 100% marks guaranteed. WhatsApp +91 90000 12345. Special discount for CS307 students!!",spam,acceptable: 1.0% | toxic: 1.4% | spam: 97.7%,93.1%
4,Pooja Nair,Doubts,"Does the reflection doc need to include the prompts we used? Mine is around 40 prompts, that seems like too much to paste.",acceptable,acceptable: 97.2% | toxic: 1.4% | spam: 1.4%,91.8%
5,Harsh Agarwal,Share your artifacts,"@Rohit your expense splitter is really clean. One suggestion: there's a rounding error when splitting ₹100 three ways, it shows 33.33 x 3 = 99.99.",acceptable,acceptable: 95.6% | toxic: 1.9% | spam: 2.5%,87.2%
6,Vikram Singh,Share your artifacts,"Rohit's project is copy-paste from GitHub, everyone knows. Shameless guy, and sir still gives him marks. Pathetic.",toxic,acceptable: 3.2% | toxic: 94.2% | spam: 2.6%,83.4%
7,Mohd. Faizan,Doubts,Bhai kisi ko Codex ka access mila? My college email isn't working for the student plan.,acceptable,acceptable: 95.2% | toxic: 2.5% | spam: 2.3%,86.2%
8,Shreya Patil,Submission & grading,"Ma'am, is a deadline extension possible? Our hostel wifi was down from Thursday night till Saturday, the warden can confirm.",acceptable,acceptable: 95.0% | toxic: 2.5% | spam: 2.5%,85.5%
9,Vivek Menon,Doubts,"I found that Claude kept adding features I didn't ask for. Telling it ""do only X, nothing else"" at the start helped a lot.",acceptable,acceptable: 93.7% | toxic: 3.5% | spam: 2.9%,81.9%
10,Free AI Deals,General,Join our Telegram group for FREE Claude Pro & ChatGPT Plus accounts 🤑 only ₹199 lifetime. t.me/aitoolsdeal_india (limited slots),spam,acceptable: 0.8% | toxic: 1.2% | spam: 98.0%,94.0%


In [8]:
from IPython.display import display, HTML
from pathlib import Path

display(HTML(Path("results-v2.html").read_text(encoding="utf-8")))

ID,Author,Thread,Comment,Classification,Probabilities,Confidence
1,Arnav Deshmukh,Doubts,"Sir, for assignment 4 can we evaluate the free Gemini or does it have to be ChatGPT? I don't have a paid plan for either.",acceptable,acceptable: 97.4% | toxic: 1.5% | spam: 1.1%,92.4%
2,Prof. Sunita Rao,Doubts,"Any free chatbot is fine. Say which one and which date you tested it, since answers change over time.",acceptable,acceptable: 95.3% | toxic: 2.1% | spam: 2.6%,86.3%
3,Lavanya Menon,Share your work,"We tested ChatGPT (free plan) on 30 questions taken straight from our college FAQ page. Overall it did better than we expected on general things like how to apply for a bonafide certificate. But it got the hostel fee wrong in 4 out of 5 tries: it kept quoting last year's amount, and when we told it the new fee it apologised and then quoted the old one again two questions later. We've put all 30 questions and its answers in a sheet if anyone wants to compare with their bot.",acceptable,acceptable: 96.5% | toxic: 2.0% | spam: 1.5%,89.8%
4,Yash Malhotra,Doubts,How many test questions are enough? We have 25 but the PDF says 'a representative set'.,acceptable,acceptable: 97.9% | toxic: 1.2% | spam: 0.9%,93.7%
5,Lavanya Menon,Doubts,"Ma'am said in class that 25-40 is fine if you cover different types: facts, procedures, opinions, and trick questions.",acceptable,acceptable: 96.5% | toxic: 2.1% | spam: 1.4%,89.8%
6,EasyMarks Academy,General,"📚 Assignment 4 READY-MADE evaluation reports available!! 40 test questions + rubric + analysis, plagiarism free. Only ₹299. WhatsApp 98000 11223",spam,acceptable: 0.7% | toxic: 0.8% | spam: 98.6%,95.8%
7,Rishi Jain,Doubts,"Ugh I'm genuinely the dumbest person in this batch. Deleted my whole .env file and lost the API key, now Cloudflare won't show it again 🤡",toxic,acceptable: 38.4% | toxic: 60.5% | spam: 1.1%,27.0%
8,Farhan Qureshi,Doubts,"@Rishi happens to everyone. Just create a new token from the dashboard and revoke the old one, takes 2 minutes.",acceptable,acceptable: 96.2% | toxic: 2.3% | spam: 1.5%,88.8%
9,Aditi Kulkarni,Share your work,"Our rubric has 4 levels for each answer: correct, partly correct, wrong, and refused. We almost left out 'refused' because we thought it would never happen, but it turned out to be important: Claude refused 3 questions about exam dates and told us to check the official notice instead. We argued about whether a refusal should count as good or bad, and in the end we scored it as good when the correct answer wasn't public, and bad when it was clearly on the website.",acceptable,acceptable: 96.5% | toxic: 2.1% | spam: 1.4%,89.8%
10,Manav Jain,Share your work,Aditi your rubric is so basic even my 10 year old cousin could make it. Stop posting things just to get sir's attention.,toxic,acceptable: 15.7% | toxic: 82.8% | spam: 1.5%,56.5%


In [ ]:
from pathlib import Path

Path("MISTAKES.md").write_text(
    "# Clef Flash — Classification Mistakes\n\n",
    encoding="utf-8"
)

Path("MISTAKES-v2.md").write_text(
    "# Clef — Classification Mistakes (Harder Dataset)\n\n",
    encoding="utf-8"
)

print("Created MISTAKES.md and MISTAKES-v2.md")

In [10]:

from pathlib import Path

# Add your own manually identified mistakes below.
mistakes = {
    "MISTAKES.md": """# Clef Flash — Mistakes

## Comment ID: 27
- Clef's label: acceptable
- My corrected label: toxic
- Reason: The comment shows women in negative light and spreads negativity

## Comment ID: 42
- Clef's label: acceptable
- My corrected label: toxic
- Reason: The comment is criticizing the course contents

## Comment ID: 69
- Clef's label: acceptable
- My corrected label: toxic
- Reason: The comment is spreading toxicity towards TA.

## Comment ID: 83
- Clef's label: acceptable
- My corrected label: toxic
- Reason: The comment is spreading hate towards another student
""",
    "MISTAKES-v2.md": """# Clef — Mistakes in the Harder Dataset

## Comment ID: 42
- Clef's label: acceptable
- My corrected label: toxic
- Reason: The comment is mocking the person.


## Comment ID: 91
- Clef's label: acceptable
- My corrected label: toxic
- Reason: The comment is telling the location of person.
"""
}

for filename, content in mistakes.items():
    Path(filename).write_text(content, encoding="utf-8")

print("Mistake log files created.")


Mistake log files created.


In [11]:

notes = """# Notes

## How to run
1. Open the Python notebook in Google Colab.
2. Enter your Cloudflare Account ID and API token securely.
3. Run the cells in order to classify comments.json using clef-flash.
4. Run the cells for comments-v2.json using clef.
5. Review the generated HTML reports.

## Tools and models
- Google Colab for Python execution.
- Cloudflare Workers AI API.
- Cloudflare Clef Flash for the original dataset.
- Cloudflare Clef for the harder dataset.
- AI assistance for understanding the API and developing code.

## Experience
I used an AI model to automate the initial classification of forum
comments and manually reviewed the outputs to identify errors.
The task demonstrated the importance of human judgement when
evaluating AI-generated moderation decisions.
"""

from pathlib import Path
Path("NOTES.md").write_text(notes, encoding="utf-8")
print("NOTES.md created.")


NOTES.md created.


In [12]:

from google.colab import files

submission_files = [
    "results.html",
    "results-v2.html",
    "results.json",
    "results-v2.json",
    "MISTAKES.md",
    "MISTAKES-v2.md",
    "NOTES.md"
]

for filename in submission_files:
    if Path(filename).exists():
        files.download(filename)
    else:
        print("Missing file:", filename)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>